In [2]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0., sr_ratio=1):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."

        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        if self.sr_ratio > 1:
            x_ = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = self.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
            x_ = self.norm(x_)
            kv = self.kv(x_).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)

        return x

def print_tensor_info(name, tensor):
    print(f"\n{name}:")
    print(f"Shape: {tensor.shape}")
    print(tensor)

def test_attention_layer():
    # Hyperparameters
    dim = 8
    num_heads = 2
    H, W = 4, 4
    batch_size = 1
    seq_len = H * W  # Flattened spatial dimensions

    # Create a simple input tensor (batch_size, seq_len, dim)
    x = torch.ones(batch_size, seq_len, dim)

    # Initialize the Attention layer
    attention_layer = Attention(dim=dim, num_heads=num_heads, qkv_bias=True)

    # Print initial input
    print("Input x:")
    print_tensor_info("Input x", x)
    
    # Compute intermediate values
    q = attention_layer.q(x).reshape(batch_size, seq_len, num_heads, dim // num_heads).permute(0, 2, 1, 3)
    print_tensor_info("Query (Q) before reshape and permute", attention_layer.q(x))
    print_tensor_info("Query (Q) after reshape and permute", q)
    
    if attention_layer.sr_ratio > 1:
        x_ = x.permute(0, 2, 1).reshape(batch_size, dim, H, W)
        x_ = attention_layer.sr(x_).reshape(batch_size, dim, -1).permute(0, 2, 1)
        x_ = attention_layer.norm(x_)
        kv = attention_layer.kv(x_).reshape(batch_size, -1, 2, num_heads, dim // num_heads).permute(2, 0, 3, 1, 4)
    else:
        kv = attention_layer.kv(x).reshape(batch_size, -1, 2, num_heads, dim // num_heads).permute(2, 0, 3, 1, 4)
    
    k, v = kv[0], kv[1]
    print_tensor_info("Key (K) before splitting", kv[0])
    print_tensor_info("Value (V) before splitting", kv[1])
    
    attn = (q @ k.transpose(-2, -1)) * attention_layer.scale
    print_tensor_info("Attention scores (Q @ K^T) before softmax", attn)
    
    attn = attn.softmax(dim=-1)
    print_tensor_info("Attention weights (after softmax)", attn)
    
    attn = attention_layer.attn_drop(attn)
    print_tensor_info("Attention weights (after dropout)", attn)
    
    x = (attn @ v).transpose(1, 2).reshape(batch_size, seq_len, dim)
    print_tensor_info("Intermediate output (after attention)", x)
    
    x = attention_layer.proj(x)
    print_tensor_info("Output (after projection)", x)
    
    x = attention_layer.proj_drop(x)
    print_tensor_info("Output (after dropout)", x)

# Run the test
test_attention_layer()


Input x:

Input x:
Shape: torch.Size([1, 16, 8])
tensor([[[1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.]]])

Query (Q) before reshape and permute:
Shape: torch.Size([1, 16, 8])
tensor([[[-0.9294,  0.2111, -0.4245, -0.0767,  0.1489, -0.3344, -0.4009,
           0.6817],
         [-0.9294,  0.2111, -0.4245, -0.0767,  0.1489, -0.3344, -0.4009,
           0.6817],
      

In [11]:
l = torch.ones(1,2,3,4)
l.T.shape

C:\Users\30744\anaconda3\envs\picClassification\lib\site-packages\ipykernel_launcher.py:2: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3281.)
  


torch.Size([4, 3, 2, 1])

In [10]:
l.reshape(2,1,3)

tensor([[[1., 1., 1.]],

        [[1., 1., 1.]]])